# Final Project for Course 3 - OMDb and Dad Jokes Mashup

This project will take you through the process of mashing up data from two different APIs.  

[The OMDb API](https://www.omdbapi.com/) lets you provide a movie title as a query input and get back data about the movie, including scores from various review sites (Rotten Tomatoes, IMDb, etc.).

[icanhazdadjokes.com](https://icanhazdadjoke.com/) returns random dad jokes containing a search string that you specify in your query.

The end result of this project will be a function that takes in a movie title as input and produces a formatted text string that includes a couple dad jokes related to a word from the movie's plot.

For example, here are a couple of sample outputs:

---

```
Baby Mama
Rotten Tomatoes rating: 63%
A successful, single businesswoman who dreams of having a baby discovers she is infertile and hires a **working** class woman to be her unlikely surrogate.
Speaking of **working**, that reminds me of some jokes.
Hope they're better than the movie!

Want to hear a joke about construction? Nah, I'm still **working** on it.
Why doesn't the Chimney-Sweep call out sick from work? Because he's used to **working** with a flue.
```

---

```
Back to the Future
Rotten Tomatoes rating: 93%
Marty McFly, a 17-year-old high school student, is **accidentally** sent 30 years into the past in a time-traveling DeLorean invented by his close friend, the maverick scientist Doc Brown.
Speaking of **accidentally**, that reminds me of some jokes.
Hope they're as good as the movie!

I **accidentally** drank a bottle of invisible ink. Now I’m in hospital, waiting to be seen.
A butcher **accidentally** backed into his meat grinder and got a little behind in his work that day.
```
---


To avoid problems with rate limits and site accessibility, we have provided a cache file with results for all the queries you need to make to both OMDb and icanhazdadjokes. Just use `requests_with_caching.get()` rather than `requests.get()`. If you're having trouble, you may not be formatting your queries properly, or you may not be asking for data that exists in our cache. We will try to provide as much information as we can to help guide you to form queries for which data exists in the cache.

## ALERT: All Query Results Should Be Found in the Cache File
If you ever run `requests_with_caching.get()` and it says either of the following, then **something was wrong** with your query:
- new; adding to cache
- found in page-specific cache


In [1]:
import requests_with_caching

In [2]:
apikey = "abcd1234"  # you may *optionally* replace this with your API key.
# Note: you do *not* need an API key to complete this assignment. Every request should be in the cache
requests_with_caching.clear_cache()
# print(list(requests_with_caching.perm_cache().keys()))

# Fetching movie info from OMDb
Your first task will be to fetch data from OMDb. The documentation for the API is at [https://www.omdbapi.com/](https://www.omdbapi.com/)

Define a function called `get_movie_data`. It takes in one parameter which is a string that should represent the title of a movie you want to search. The function should return a dictionary with information about that movie.

Again, use `requests_with_caching.get()`. If you were to use the live OMDP API, you would need to get an API key, as described in the documentation. However **you do not need an API key** to complete this assignment. For the queries on movies that are already in the permanent cache, you won’t need an API key. 

HINT: Be sure to include **only** keys `t` and `r` as query parameters in order to extract data from the cache. If any other parameters are included,  the function will not be able to recognize the data that you're attempting to pull from the cache.

The following movie titles are in the cache:
- Black Panther
- Venom
- Baby Mama
- Sherlock Holmes
- Return of the Jedi
- Back to the Future

In [3]:
# NOTE: the OMDb API uses http:// instead of https://
import requests_with_caching as cache
import json

def get_movie_data(name: str) -> dict:
    """Returns a dictionary of movie information from the OMDb API.

    Parameters
    ----------
    name : str
        The name of the movie to search for.

    Returns
    -------
    dict
        A dictionary of movie information.
    """
    movie_param = {'r':'json', 't': name}
    url = "http://www.omdbapi.com/"
    results = cache.get(url, params=movie_param)
    return dict(json.loads(results.text))

In [4]:
results = get_movie_data("Black Panther")
assert type(results) == type({})
assert results["Year"] == "2018"

# some other invocations that we use in the automated tests; uncomment these if you are getting errors and want better error messages
# print(get_movie_data("Venom"))
# print(get_movie_data("Baby Mama"))

found in permanent_cache


## Extract the Rotten Tomatoes Rating for a Movie
Next, you will write a function that extracts the Rotten Tomatoes rating for a movie from the results dictionary as an *integer*. If the movie does not have a Rotten Tomatoes rating, return `-1`.

Fill in the template for the function below

In [5]:
def rt_rating(movie_data: dict) -> int:
    """Returns the Rotten Tomatoes rating from a dictionary of movie information.

    Parameters
    ----------
    movie_data : dict
        A dictionary of movie information.

    Returns
    -------
    int
        The Rotten Tomatoes rating. For example, 75% would be returned as the integer 75.
        -1 if no records found.
    """
    try:
        return int(dict(list(d for d in movie_data['Ratings'] if d['Source'] == 'Rotten Tomatoes')[0])['Value'].replace('%',''))
    except:
        return -1 

# We suggest that you write an assert statement to check the output of your function for the movie Black Panther. The autograder will check results for some other movies.
results = get_movie_data("Black Panther")
#results = get_movie_data("Eyyvah Eyvah")
print(rt_rating(results))
#assert rt_rating(results) == 96

found in permanent_cache
96


In [18]:
def m_plot(movie_data: dict) -> str:
    """Returns the Rotten Tomatoes plot from a dictionary of movie information.

    Parameters
    ----------
    movie_data : dict
        A dictionary of movie information.

    Returns
    -------
    str
        Plot.
    """
    return movie_data['Plot']

results = get_movie_data("Black Panther")
assert m_plot(results) == "T'Challa, heir to the hidden but advanced kingdom of Wakanda, must step forward to lead his people into a new future and must confront a challenger from his country's past."

found in permanent_cache


In [25]:
def rt_rating_evaluation(rating: int) -> str:
    """Returns the Rotten Tomatoes rating evaluation information.
    
    - No Rotten Tomates Rating (meaning the rating is `-1`): `"Hope you like them!"`
    - Rotten Tomatoes Rating below 70%: `"Hope they're better than the movie!"`
    - Rotten Tomates 70%+: `"Hope they're as good as the movie!"`

    Parameters
    ----------
    rating : int
        Rating information.

    Returns
    -------
    str
        Evaluation.
    """
    return_val = ""
    if rating == -1:
        return_val = "Hope you like them!"
    elif rating < 70:
        return_val = "Hope they're better than the movie!"
    else:
        return_val = "Hope they're as good as the movie!"
    return return_val

print(rt_rating_evaluation(100))

Hope they're as good as the movie!


# Fetching Jokes
Now you will use another API to fetch a couple of dad jokes related to a movie's plot.

You will do this in two stages. First you'll implement a helper function that calls the API to get jokes, asking for jokes related to a single word.

Then you'll use that helper function, calling it with the longest words from the plot summary until it finds one that there are jokes for.


## Search for Jokes Containing a Word
To search for dad jokes, you'll be using the API for icanhazdadjokes. The documentation for the API is at [https://icanhazdadjoke.com/api](https://icanhazdadjoke.com/api)

Define a function called `get_joke_data`. It takes in one parameter which is a string. The function should return a dictionary with information about **up to two** jokes that contain that string.

Again, use `requests_with_caching.get()`. All the query results you need are already in the permanent cache.

- Note 1: Be sure to include **only** keys `term` and `limit` as query parameters in order to extract data from the cache. If any other parameters are included, the function will not be able to recognize the data that you're attempting to pull from the cache.
- Note 2: Use the `limit` parameter in the icanhazdadjokes API to limit it to two results (instead of slicing)

In [7]:
# no template is provided for this function. You have to define it from scratch
import requests_with_caching as cache
import json

def get_joke_data(word: str) -> dict:
    """Returns a dictionary of jokes information from the API.

    Parameters
    ----------
    word : str
        The word to search for.

    Returns
    -------
    dict
        A dictionary of jokes information.
    """
    joke_param = {'limit': 2, 'term': word.lower()}
    url = "https://icanhazdadjoke.com/search"
    results = cache.get(url, params=joke_param)
    return_val: dict = json.loads(results.text) if not results.text.startswith("<!DOCTYPE html>") else None
    return return_val

In [8]:
print(get_joke_data("attended_for_the_first_time"))
print(get_joke_data("What's"))
print(get_joke_data("Class"))

found in permanent_cache
{'current_page': 1, 'limit': 2, 'next_page': 1, 'previous_page': 1, 'results': [], 'search_term': 'attended_for_the_first_time', 'status': 200, 'total_jokes': 0, 'total_pages': 1}
new; adding to cache
None
found in permanent_cache
{'current_page': 1, 'limit': 2, 'next_page': 1, 'previous_page': 1, 'results': [{'id': 'usrcaMuszd', 'joke': "What's the worst thing about ancient history class? The teachers tend to Babylon."}], 'search_term': 'class', 'status': 200, 'total_jokes': 1, 'total_pages': 1}


In [9]:
assert (
    len(get_joke_data("magic")["results"]) == 2
), "The correct number of jokes for 'magic' should be 2"
assert (
    get_joke_data("magic")["results"][0]["joke"]
    == "What do you call a magician who has lost their magic? Ian."
)

found in permanent_cache
found in permanent_cache


## Get Jokes for a Long Word from the Plot Description

Now you'll define a function called `get_jokes`. It will extract the longest word from the plot and try to find jokes for it. If there aren't any, it will proceed to the next longest word, and so on, until it finds a word for which there are jokes. If there is more than one word of the same length, try words that are earlier in the description first (which `sorted` does by default, since it's "stable" and will only move things around the minimum necessary).

We provide code that removes punctuation from the words in `plot` and assigns the result to the variable `words`. Your code should extend this by sorting `words` from longest to shortest and use the sorted list (and the `get_joke_data` function that you defined above) to find the longest word with associated jokes. If there are no words with associated jokes, your function should return the tuple `(None, None)`. If there is a word with associated jokes, your function should return a tuple with (1) the longest word with a joke and (2) a list of jokes associated with that word (as a list of strings).

In [10]:
def get_jokes(plot: str, verbosity=0) -> tuple[str, list[str]]:
    """Returns a tuple containing the longest word for which jokes were found
    and the joke itself. Break ties for longest word using the order in `plot`.
    Make sure that you strip punctuation from the word before you search for a joke.

    Parameters
    ----------
    plot : str
        The plot of a movie.

    verbosity : int (optional)
        If 0, no output is printed. If 1, some output is printed about which words were tried.
        Defaults to 0.

    Returns
    -------
    tuple[str, list[str]]
        A tuple containing the word that was used to search for a joke and a list of two joke strings.
    """

    words = plot.split()  # split into separate words
    words = [w.strip(",.!;:") for w in words]  # remove punctuation for each word

    # WRITE YOUR CODE HERE
    words = [w.strip("?") for w in words]
    if verbosity: print(words)
    
    lengths = [len(w) for w in words]    
    if verbosity: print(lengths)
    
    words_length = list(zip(words, lengths))   
    if verbosity: print(words_length)
    
    words_length_sorted = sorted(words_length, key=lambda k: k[1], reverse=True)    
    if verbosity: print(words_length_sorted)
    
    return_val: tuple[str, list[str]] = (None, None)
    
    for mw in words_length_sorted:
        max_word_length = mw[0]
        if verbosity: print(max_word_length)

        word_jokes = get_joke_data(max_word_length)
        if verbosity: print(word_jokes)

        if word_jokes is not None:
            word_jokes = [joke["joke"] for joke in get_joke_data(max_word_length)["results"]]             
            if len(word_jokes) > 0:
                if verbosity: print(word_jokes)
                return_val = (max_word_length, word_jokes)
                break
            
    if verbosity: print(return_val)
    
    return return_val

In [11]:
# Test case 1: Plot with first word that has jokes
plot = "I had dreams of a cat."
result = get_jokes(plot, 1)
assert result[0] == "dreams"
assert (
    result[1][0]
    == "I'm tired of following my dreams. I'm just going to ask them where they are going and meet up with them later."
)

['I', 'had', 'dreams', 'of', 'a', 'cat']
[1, 3, 6, 2, 1, 3]
[('I', 1), ('had', 3), ('dreams', 6), ('of', 2), ('a', 1), ('cat', 3)]
[('dreams', 6), ('had', 3), ('cat', 3), ('of', 2), ('I', 1), ('a', 1)]
dreams
found in permanent_cache
{'current_page': 1, 'limit': 2, 'next_page': 1, 'previous_page': 1, 'results': [{'id': '0189hNRf2g', 'joke': "I'm tired of following my dreams. I'm just going to ask them where they are going and meet up with them later."}], 'search_term': 'dreams', 'status': 200, 'total_jokes': 1, 'total_pages': 1}
found in permanent_cache
["I'm tired of following my dreams. I'm just going to ask them where they are going and meet up with them later."]
('dreams', ["I'm tired of following my dreams. I'm just going to ask them where they are going and meet up with them later."])


In [12]:
# Test case 2: Plot with a long word that hasn't jokes but second does
plot = "The cat attended_for_the_first_time the class."
result = get_jokes(plot, 1)
assert result == ('class', ["What's the worst thing about ancient history class? The teachers tend to Babylon."])

['The', 'cat', 'attended_for_the_first_time', 'the', 'class']
[3, 3, 27, 3, 5]
[('The', 3), ('cat', 3), ('attended_for_the_first_time', 27), ('the', 3), ('class', 5)]
[('attended_for_the_first_time', 27), ('class', 5), ('The', 3), ('cat', 3), ('the', 3)]
attended_for_the_first_time
found in permanent_cache
{'current_page': 1, 'limit': 2, 'next_page': 1, 'previous_page': 1, 'results': [], 'search_term': 'attended_for_the_first_time', 'status': 200, 'total_jokes': 0, 'total_pages': 1}
found in permanent_cache
class
found in permanent_cache
{'current_page': 1, 'limit': 2, 'next_page': 1, 'previous_page': 1, 'results': [{'id': 'usrcaMuszd', 'joke': "What's the worst thing about ancient history class? The teachers tend to Babylon."}], 'search_term': 'class', 'status': 200, 'total_jokes': 1, 'total_pages': 1}
found in permanent_cache
["What's the worst thing about ancient history class? The teachers tend to Babylon."]
('class', ["What's the worst thing about ancient history class? The teache

In [13]:
# Test case 3: Plot with words that hasn't jokes but eight does
plot = "What's the worst thing about ancient history class? The teachers tend to Babylon."
result = get_jokes(plot, 1)
assert result == ('class', ["What's the worst thing about ancient history class? The teachers tend to Babylon."])

["What's", 'the', 'worst', 'thing', 'about', 'ancient', 'history', 'class', 'The', 'teachers', 'tend', 'to', 'Babylon']
[6, 3, 5, 5, 5, 7, 7, 5, 3, 8, 4, 2, 7]
[("What's", 6), ('the', 3), ('worst', 5), ('thing', 5), ('about', 5), ('ancient', 7), ('history', 7), ('class', 5), ('The', 3), ('teachers', 8), ('tend', 4), ('to', 2), ('Babylon', 7)]
[('teachers', 8), ('ancient', 7), ('history', 7), ('Babylon', 7), ("What's", 6), ('worst', 5), ('thing', 5), ('about', 5), ('class', 5), ('tend', 4), ('the', 3), ('The', 3), ('to', 2)]
teachers
new; adding to cache
None
ancient
new; adding to cache
None
history
new; adding to cache
None
Babylon
new; adding to cache
None
What's
found in page-specific cache
None
worst
new; adding to cache
None
thing
new; adding to cache
None
about
new; adding to cache
None
class
found in permanent_cache
{'current_page': 1, 'limit': 2, 'next_page': 1, 'previous_page': 1, 'results': [{'id': 'usrcaMuszd', 'joke': "What's the worst thing about ancient history class? The

BONUS CHALLENGE (ungraded). If we had specified that ties should be broken by taking words in the alphabetic order rather than later, how could you have done that? Try writing a test and then implementing it!

## Put it All Together

Now put it all together to make the full app. Define a function, `haha_me`. It takes a movie name as input and verbosity and returns a text string that is meant to entertain the reader.

We have provided a helper function, `highlight`, that highlights a string inside a larger string by wrapping it in asterisks (`*`). Try invoking it a few times to make sure you understand what it does, then figure out how it should be used based on the sample outputs in the assert statements.

If the movie is not found in the OMDb API (using `get_movie_data`), return `"No movie found: "` followed by the movie title.

If the movie is found, but there are no jokes (`get_jokes` returned `(None, None)`), return `"I've got no jokes about this movie. It's too serious!"`.

If the movie and jokes are found, your function should return a string with each of the following on separate lines:
- The name of the movie
- `"Rotten Tomatoes rating: XX%"` (replacing `"XX"` with the actual Rotten Tomatoes rating)
- The plot of the movie with the joke keyword highlighted (using the provided `highlight` function)
- `"Speaking of **YY**, that reminds me of some jokes."` (replacing `"YY"` with the joke keyword)
- A different phrase about the jokes, depending on the Rotten Tomatoes rating:
    - No Rotten Tomates Rating (meaning the rating is `-1`): `"Hope you like them!"`
    - Rotten Tomatoes Rating below 70%: `"Hope they're better than the movie!"`
    - Rotten Tomates 70%+: `"Hope they're as good as the movie!"`
- *(an empty line)*
- The list of jokes, separated by a newline (using `"\n".join(...)`)

For example, for Venom:
```
Venom
Rotten Tomatoes rating: 30%
A failed reporter is bonded to an alien entity, one of many symbiotes who have invaded **Earth**. But the being takes a liking to **Earth** and decides to protect it.
Speaking of **Earth**, that reminds me of some jokes.
Hope they're better than the movie!

Astronomers got tired watching the moon go around the **Earth** for 24 hours. They decided to call it a day.
The rotation of **Earth** really makes my day.
```

In [14]:
def highlight(word: str, sentence: str) -> str:
    """
    Highlights a specific word in a sentence by surrounding it with asterisks (**).
    The highlighting is case-insensitive.

    Args:
        word (str): The word to be highlighted.
        sentence (str): The sentence in which the word should be highlighted.

    Returns:
        str: The sentence with the specified word highlighted.
    """
    import re

    # Escape special characters in the word to treat it as a literal string
    # Use re.sub() to replace the word with the highlighted version
    # Use re.IGNORECASE flag to perform case-insensitive replacement
    return re.sub(re.escape(word), f"**{word}**", sentence, flags=re.IGNORECASE)

In [26]:
def haha_me(movie_title: str, verbosity=0) -> str:
    """Returns a string containing a movie title, the Rotten Tomatoes rating, a plot summary,
    and two jokes about the movie. If no jokes are found, a message is returned instead.

    Parameters
    ----------
    movie_title : str
        The title of a movie.

    verbosity : int (optional)
        If 0, no output is printed. If 1, some output is printed about which plot words were tried.
        Defaults to 0.
    """
    NEW_LINE = "\n"
    data = get_movie_data(movie_title)
    rating = rt_rating(data)
    movie_plot = m_plot(data)
    jokes = get_jokes(movie_plot, verbosity)
    jokes_word = jokes[0]
    if verbosity: print(jokes_word)
    return_val = movie_title + NEW_LINE + \
                "Rotten Tomatoes rating: XX%".replace("XX", str(rating)) + NEW_LINE + \
                movie_plot + NEW_LINE + \
                "Speaking of YY, that reminds me of some jokes.".replace("YY", jokes_word) + NEW_LINE + \
                rt_rating_evaluation(rating) + NEW_LINE + \
                NEW_LINE + \
                "\n".join(jokes[1])
    return_val = highlight(jokes_word, return_val)
    if verbosity: print(return_val)
    return return_val

print(haha_me("Black Panther", 1))

found in permanent_cache
["T'Challa", 'heir', 'to', 'the', 'hidden', 'but', 'advanced', 'kingdom', 'of', 'Wakanda', 'must', 'step', 'forward', 'to', 'lead', 'his', 'people', 'into', 'a', 'new', 'future', 'and', 'must', 'confront', 'a', 'challenger', 'from', 'his', "country's", 'past']
[8, 4, 2, 3, 6, 3, 8, 7, 2, 7, 4, 4, 7, 2, 4, 3, 6, 4, 1, 3, 6, 3, 4, 8, 1, 10, 4, 3, 9, 4]
[("T'Challa", 8), ('heir', 4), ('to', 2), ('the', 3), ('hidden', 6), ('but', 3), ('advanced', 8), ('kingdom', 7), ('of', 2), ('Wakanda', 7), ('must', 4), ('step', 4), ('forward', 7), ('to', 2), ('lead', 4), ('his', 3), ('people', 6), ('into', 4), ('a', 1), ('new', 3), ('future', 6), ('and', 3), ('must', 4), ('confront', 8), ('a', 1), ('challenger', 10), ('from', 4), ('his', 3), ("country's", 9), ('past', 4)]
[('challenger', 10), ("country's", 9), ("T'Challa", 8), ('advanced', 8), ('confront', 8), ('kingdom', 7), ('Wakanda', 7), ('forward', 7), ('hidden', 6), ('people', 6), ('future', 6), ('heir', 4), ('must', 4), (

In [27]:
assert (
    haha_me("Black Panther")
    == """Black Panther
Rotten Tomatoes rating: 96%
T'Challa, heir to the hidden but advanced kingdom of Wakanda, must step **forward** to lead his people into a new future and must confront a challenger from his country's past.
Speaking of **forward**, that reminds me of some jokes.
Hope they're as good as the movie!

Sometimes I tuck my knees into my chest and lean **forward**.  That’s just how I roll.
Why do scuba divers fall backwards into the water? Because if they fell **forward**s they’d still be in the boat."""
)

found in permanent_cache
found in permanent_cache
found in permanent_cache
found in permanent_cache
found in permanent_cache
found in page-specific cache
found in permanent_cache
found in permanent_cache
found in permanent_cache
found in permanent_cache
found in permanent_cache
found in permanent_cache
found in page-specific cache
found in permanent_cache
found in permanent_cache


In [28]:
print(get_movie_data("El Pescador"))

new; adding to cache
{'Response': 'False', 'Error': 'No API key provided.'}
